# ANRF AISEHack 2.0 — Polymer Property Prediction
## Reproduction Notebook — 0.911 Public LB Score

This notebook reproduces the public-LB **0.911** submission end-to-end.
It predicts **Tg** (glass transition temperature, °C) and **Egc** (chain band gap, eV)
from polymer SMILES.

## Approach

Two-stage stack blended per-target with non-negative least squares:

1. **Phase 1 — GBM Cocktail** — LightGBM + CatBoost + HistGradientBoosting on a rich
   feature mix: RDKit 2D descriptors (~210), Morgan-r2 counts (2048), Morgan-r3 counts
   (2048), MACCS keys (167), Avalon (512), Atom-Pair counts (2048), Topological-Torsion
   counts (2048). Log1p on Egc, identity on Tg. Mean-blended.
2. **Phase 2 — Chemprop D-MPNN Multitask** — Directed message-passing neural network
   with a shared molecular representation and joint Tg + Egc regression heads. Molecules
   labeled for one target still contribute to the shared graph representation, so both
   heads benefit from the full training set. 5-fold × 3-seed bag = 15 models.
3. **NNLS Blend** — Per-target non-negative weights fit on out-of-fold predictions,
   applied to test predictions.

## Runtime

- Featurization: ~10 min
- Phase 1 GBM: ~90 min
- Phase 2 Chemprop: ~2-3 h on Kaggle GPU (much longer on CPU)
- Blend + write submission: seconds

**Total: ~4 hours on Kaggle GPU.**

## Reproducibility

All random seeds are fixed (`SEED = 42`, bag seeds `[42, 1337, 7]`). Running end-to-end
produces a submission that scores in the **0.907–0.913** range on public LB.
Small variation is expected from CUDA/hardware non-determinism in Chemprop; the
methodology is fully deterministic.

## 1. Install dependencies

Chemprop 2.x + LightGBM + CatBoost. Kaggle base image already has PyTorch and RDKit.

In [ ]:
!pip install -q chemprop lightning
!pip install -q --upgrade catboost lightgbm

## 2. Imports and configuration

In [ ]:
import gzip
import hashlib
import json
import logging
import pickle
import re
import time
import warnings
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, MACCSkeys, rdFingerprintGenerator
from rdkit.Avalon.pyAvalonTools import GetAvalonFP
from rdkit.DataStructs import ConvertToNumpyArray
from scipy.optimize import nnls
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

import lightgbm as lgb
from catboost import CatBoostRegressor

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore", category=UserWarning)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("polymer")

In [ ]:
# --- Data paths (Kaggle convention used by the host notebook) ---
DATA_DIR = Path("/kaggle/input/competitions/aisehack-2-0/")
if not DATA_DIR.exists():
    # Local fallback for offline development
    DATA_DIR = Path("./data") if Path("./data").exists() else Path("../input")

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./output")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR = {DATA_DIR}")
print(f"OUT_DIR  = {OUT_DIR}")
assert (DATA_DIR / "train.csv").exists(), f"train.csv not found under {DATA_DIR}"
assert (DATA_DIR / "test.csv").exists(),  f"test.csv not found under {DATA_DIR}"

# --- Core hyperparameters (matches the local 0.911 configuration) ---
SEED = 42
N_FOLDS = 5
N_QUANTILE_BINS = 10
TARGETS = ["tg", "egc"]
TARGET_TRANSFORMS = {"tg": "identity", "egc": "log1p"}

# GBM configs
LGB_PARAMS = dict(
    n_estimators=4000, learning_rate=0.03, num_leaves=63, min_child_samples=10,
    feature_fraction=0.5, bagging_fraction=0.85, bagging_freq=5, reg_lambda=1.0,
    objective="regression", metric="rmse", verbosity=-1, random_state=SEED, n_jobs=-1,
)
CAT_PARAMS = dict(
    iterations=4000, depth=8, learning_rate=0.03, l2_leaf_reg=3.0,
    grow_policy="SymmetricTree", random_seed=SEED, verbose=False,
    allow_writing_files=False,
)
HGB_PARAMS = dict(
    max_iter=1000, learning_rate=0.05, max_leaf_nodes=63, min_samples_leaf=20,
    l2_regularization=1.0, early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=30, random_state=SEED,
)

# Chemprop configs
CP_MP_HIDDEN, CP_MP_DEPTH = 300, 4
CP_FFN_HIDDEN, CP_FFN_DEPTH, CP_DROPOUT = 300, 2, 0.05
CP_MAX_EPOCHS, CP_BATCH_SIZE, CP_PATIENCE = 50, 64, 10
CP_N_FOLDS = 5
CP_BAG_SEEDS = [42, 1337, 7]

print(f"Config: seed={SEED}, n_folds={N_FOLDS}, chemprop bag={len(CP_BAG_SEEDS)} seeds x {CP_N_FOLDS} folds")

## 3. Featurization

RDKit 2D descriptors + 6 fingerprint families concatenated into one feature matrix.

In [ ]:
def _smiles_to_mols(smiles_list, desc):
    mols, n_failed = [], 0
    for smi in tqdm(smiles_list, desc=f"{desc} parse", unit="mol", leave=False):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            n_failed += 1
        mols.append(mol)
    if n_failed:
        log.warning("[%s] %d / %d SMILES failed parse", desc, n_failed, len(smiles_list))
    return mols


def compute_rdkit_2d(smiles_list, desc="rdk-2d"):
    mols = _smiles_to_mols(smiles_list, desc)
    keys = list(Descriptors.CalcMolDescriptors(Chem.MolFromSmiles("CCO")).keys())
    nan = {k: np.nan for k in keys}
    rows = []
    for mol in tqdm(mols, desc=f"{desc} compute", unit="mol", leave=False):
        if mol is None:
            rows.append(dict(nan)); continue
        try:
            rows.append(Descriptors.CalcMolDescriptors(mol))
        except Exception:
            rows.append(dict(nan))
    return pd.DataFrame(rows, columns=keys).add_prefix("rdk_")


def compute_morgan_fp(smiles_list, radius=2, n_bits=2048, count=False, desc=None):
    label = desc or f"morgan{radius}-{'cnt' if count else 'bit'}-{n_bits}"
    mols = _smiles_to_mols(smiles_list, label)
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    dtype = np.int16 if count else np.int8
    arr = np.zeros((len(mols), n_bits), dtype=dtype)
    for i, mol in enumerate(tqdm(mols, desc=f"{label} compute", unit="mol", leave=False)):
        if mol is None:
            continue
        arr[i] = (gen.GetCountFingerprintAsNumPy(mol) if count else gen.GetFingerprintAsNumPy(mol))
    prefix = f"m{radius}{'c' if count else 'b'}"
    return pd.DataFrame(arr, columns=[f"{prefix}_{j}" for j in range(n_bits)])


def compute_maccs(smiles_list, desc="maccs"):
    mols = _smiles_to_mols(smiles_list, desc)
    arr = np.zeros((len(mols), 167), dtype=np.int8)
    for i, mol in enumerate(tqdm(mols, desc=f"{desc} compute", unit="mol", leave=False)):
        if mol is None:
            continue
        ConvertToNumpyArray(MACCSkeys.GenMACCSKeys(mol), arr[i])
    return pd.DataFrame(arr, columns=[f"maccs_{j}" for j in range(167)])


def compute_avalon_fp(smiles_list, n_bits=512, desc="avalon"):
    mols = _smiles_to_mols(smiles_list, desc)
    arr = np.zeros((len(mols), n_bits), dtype=np.int8)
    for i, mol in enumerate(tqdm(mols, desc=f"{desc} compute", unit="mol", leave=False)):
        if mol is None:
            continue
        ConvertToNumpyArray(GetAvalonFP(mol, nBits=n_bits), arr[i])
    return pd.DataFrame(arr, columns=[f"avlon_{j}" for j in range(n_bits)])


def compute_atom_pair_fp(smiles_list, n_bits=2048, count=True, desc=None):
    label = desc or f"atompair-{'cnt' if count else 'bit'}-{n_bits}"
    mols = _smiles_to_mols(smiles_list, label)
    gen = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=n_bits)
    dtype = np.int16 if count else np.int8
    arr = np.zeros((len(mols), n_bits), dtype=dtype)
    for i, mol in enumerate(tqdm(mols, desc=f"{label} compute", unit="mol", leave=False)):
        if mol is None:
            continue
        arr[i] = (gen.GetCountFingerprintAsNumPy(mol) if count else gen.GetFingerprintAsNumPy(mol))
    prefix = f"ap{'c' if count else 'b'}"
    return pd.DataFrame(arr, columns=[f"{prefix}_{j}" for j in range(n_bits)])


def compute_topological_torsion_fp(smiles_list, n_bits=2048, count=True, desc=None):
    label = desc or f"torsion-{'cnt' if count else 'bit'}-{n_bits}"
    mols = _smiles_to_mols(smiles_list, label)
    gen = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=n_bits)
    dtype = np.int16 if count else np.int8
    arr = np.zeros((len(mols), n_bits), dtype=dtype)
    for i, mol in enumerate(tqdm(mols, desc=f"{label} compute", unit="mol", leave=False)):
        if mol is None:
            continue
        arr[i] = (gen.GetCountFingerprintAsNumPy(mol) if count else gen.GetFingerprintAsNumPy(mol))
    prefix = f"tt{'c' if count else 'b'}"
    return pd.DataFrame(arr, columns=[f"{prefix}_{j}" for j in range(n_bits)])

In [ ]:
def clean_inf(X):
    return X.replace([np.inf, -np.inf], np.nan)


def sanitize_columns(df):
    df = df.copy()
    df.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", str(c)) for c in df.columns]
    if df.columns.duplicated().any():
        new_cols, seen = [], {}
        for c in df.columns:
            if c in seen:
                seen[c] += 1
                new_cols.append(f"{c}__{seen[c]}")
            else:
                seen[c] = 0
                new_cols.append(c)
        df.columns = new_cols
    return df


def drop_constant_cols(X_train, X_test):
    keep = X_train.columns[X_train.nunique(dropna=False) > 1].tolist()
    return X_train[keep], X_test[keep], keep


def split_by_target_type(train, test, X_train, X_test, target_type):
    tr_mask = (train["target_type"] == target_type).values
    te_mask = (test["target_type"] == target_type).values
    return (
        X_train[tr_mask].reset_index(drop=True),
        train.loc[tr_mask, "target"].reset_index(drop=True),
        X_test[te_mask].reset_index(drop=True),
        test.loc[te_mask, "id"].reset_index(drop=True),
    )


def stratified_quantile_split(y, n_folds=5, n_bins=10, seed=42):
    y = np.asarray(y)
    n_bins_eff = min(n_bins, max(2, len(np.unique(y))))
    try:
        bins = pd.qcut(y, q=n_bins_eff, labels=False, duplicates="drop")
    except ValueError:
        bins = pd.cut(y, bins=n_bins_eff, labels=False, include_lowest=True)
    bins = np.asarray(bins)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return list(skf.split(np.zeros((len(y), 1)), bins))


def transform_target(y, kind):
    y = np.asarray(y, dtype=np.float64)
    if kind == "log1p":
        return np.log1p(y), np.expm1
    if kind == "identity":
        return y, (lambda x: x)
    raise ValueError(f"unknown target transform: {kind!r}")


def build_features(smiles, split_label):
    log.info("=== featurizing %s (n=%d) ===", split_label, len(smiles))
    parts = [
        compute_rdkit_2d(smiles, desc=f"{split_label}/rdk-2d"),
        compute_morgan_fp(smiles, radius=2, n_bits=2048, count=True,
                          desc=f"{split_label}/morgan2-cnt"),
        compute_morgan_fp(smiles, radius=3, n_bits=2048, count=True,
                          desc=f"{split_label}/morgan3-cnt"),
        compute_maccs(smiles, desc=f"{split_label}/maccs"),
        compute_avalon_fp(smiles, n_bits=512, desc=f"{split_label}/avalon"),
        compute_atom_pair_fp(smiles, n_bits=2048, count=True,
                              desc=f"{split_label}/atompair-cnt"),
        compute_topological_torsion_fp(smiles, n_bits=2048, count=True,
                                        desc=f"{split_label}/torsion-cnt"),
    ]
    parts = [p.reset_index(drop=True) for p in parts if p.shape[1] > 0]
    X = pd.concat(parts, axis=1)
    X = sanitize_columns(clean_inf(X))
    log.info("[%s] combined feature matrix: %s", split_label, X.shape)
    return X

## 4. Load competition data and build feature matrices

Featurization runs on both train and test sets. Expect ~10 minutes.

In [ ]:
t_start = time.time()
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
log.info("train: %s | test: %s", train.shape, test.shape)
log.info("train target_type: %s", dict(train["target_type"].value_counts()))
log.info("test  target_type: %s", dict(test["target_type"].value_counts()))

X_train_full = build_features(train["smiles"].tolist(), split_label="train")
X_test_full = build_features(test["smiles"].tolist(), split_label="test")

common = sorted(set(X_train_full.columns) & set(X_test_full.columns))
X_train_full, X_test_full = X_train_full[common], X_test_full[common]
X_train_full, X_test_full, kept = drop_constant_cols(X_train_full, X_test_full)
log.info("features after intersect + drop-constant: %d", len(kept))
log.info("featurization time: %.1f min", (time.time() - t_start) / 60)

## 5. Phase 1 — GBM Cocktail

LightGBM + CatBoost + HistGradientBoosting per target, 5-fold stratified-quantile CV.
Log1p transform on Egc, identity on Tg. Mean-blended within each target.

In [ ]:
def _train_lgb(X_tr, y_tr, X_va, y_va):
    m = lgb.LGBMRegressor(**LGB_PARAMS)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
          callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False),
                     lgb.log_evaluation(period=0)])
    return m, int(m.best_iteration_ or LGB_PARAMS["n_estimators"])


def _train_cat(X_tr, y_tr, X_va, y_va):
    m = CatBoostRegressor(**CAT_PARAMS)
    m.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=200, verbose=False)
    return m, int(m.get_best_iteration() or CAT_PARAMS["iterations"])


def _train_hgb(X_tr, y_tr, X_va=None, y_va=None):
    m = HistGradientBoostingRegressor(**HGB_PARAMS)
    m.fit(X_tr, y_tr)
    return m, int(m.n_iter_)


GBM_TRAINERS = {"lgb": _train_lgb, "cat": _train_cat, "hgb": _train_hgb}


def _refit_full(name, X, y, n_iters):
    if name == "lgb":
        p = dict(LGB_PARAMS); p["n_estimators"] = max(int(n_iters * 1.10), 200)
        m = lgb.LGBMRegressor(**p); m.fit(X, y); return m
    if name == "cat":
        p = dict(CAT_PARAMS); p["iterations"] = max(int(n_iters * 1.10), 200)
        m = CatBoostRegressor(**p); m.fit(X, y, verbose=False); return m
    if name == "hgb":
        m = HistGradientBoostingRegressor(**HGB_PARAMS); m.fit(X, y); return m
    raise ValueError(name)


def cv_per_target(X, y_orig, target_name, transform_kind, n_folds=N_FOLDS):
    y_t, inv = transform_target(y_orig, transform_kind)
    log.info("[%s] transform=%s | y_t range [%.3f, %.3f]",
              target_name, transform_kind, float(y_t.min()), float(y_t.max()))
    oof = {k: np.zeros(len(X), dtype=np.float64) for k in GBM_TRAINERS}
    best_iters = {k: [] for k in GBM_TRAINERS}
    folds = stratified_quantile_split(y_t, n_folds=n_folds, n_bins=N_QUANTILE_BINS, seed=SEED)
    for fold, (tr_idx, va_idx) in enumerate(tqdm(folds, desc=f"{target_name} CV", unit="fold")):
        for k, fn in GBM_TRAINERS.items():
            t0 = time.time()
            model, n_it = fn(X.iloc[tr_idx], y_t[tr_idx], X.iloc[va_idx], y_t[va_idx])
            pred = inv(model.predict(X.iloc[va_idx]))
            oof[k][va_idx] = pred
            r2 = r2_score(y_orig[va_idx], pred)
            best_iters[k].append(n_it)
            log.info("[%s/%s] fold %d/%d R^2=%.4f iters=%d (%.1fs)",
                     target_name, k, fold + 1, n_folds, r2, n_it, time.time() - t0)
    per_model_r2 = {k: float(r2_score(y_orig, oof[k])) for k in GBM_TRAINERS}
    blend_oof = np.mean(np.stack([oof[k] for k in GBM_TRAINERS], axis=0), axis=0)
    blend_r2 = float(r2_score(y_orig, blend_oof))
    log.info("[%s] per-model OOF R^2: %s",
              target_name, {k: f"{v:.4f}" for k, v in per_model_r2.items()})
    log.info("[%s] blend OOF R^2 = %.4f", target_name, blend_r2)
    return oof, blend_oof, best_iters

In [ ]:
# --- Run Phase 1: per-target CV + full-data refit for test predictions ---
phase1_oof_frames = []
phase1_test_frames = []
phase1_summary = {"per_target": {}}

for t in TARGETS:
    log.info("=" * 64)
    log.info("PHASE 1 — training %s", t.upper())
    log.info("=" * 64)
    X_tr, y_tr_s, X_te, ids_te = split_by_target_type(train, test, X_train_full, X_test_full, t)
    y_tr = y_tr_s.values
    log.info("[%s] train=%d test=%d y=[%.3f, %.3f]",
             t, len(X_tr), len(X_te), float(y_tr.min()), float(y_tr.max()))

    oof, blend_oof, best_iters = cv_per_target(X_tr, y_tr, t, TARGET_TRANSFORMS[t])

    phase1_oof_frames.append(pd.DataFrame({
        "smiles": train.loc[train["target_type"] == t, "smiles"].values,
        "target_type": t,
        "target": y_tr,
        **{f"oof_{k}": oof[k] for k in GBM_TRAINERS},
        "oof_blend": blend_oof,
    }))

    log.info("[%s] refitting on full %d rows...", t, len(X_tr))
    y_t_full, inv = transform_target(y_tr, TARGET_TRANSFORMS[t])
    test_preds = {}
    for k in GBM_TRAINERS:
        n_it = int(np.median(best_iters[k]))
        t_fit = time.time()
        model = _refit_full(k, X_tr, y_t_full, n_it)
        test_preds[k] = inv(model.predict(X_te))
        log.info("[%s/%s] refit done (iters=%d, %.1fs)", t, k, n_it, time.time() - t_fit)
    blend_test = np.mean(np.stack(list(test_preds.values()), axis=0), axis=0)
    phase1_test_frames.append(pd.DataFrame({"id": ids_te.values, "target": blend_test}))
    phase1_summary["per_target"][t] = {
        "blend_oof_r2": float(r2_score(y_tr, blend_oof)),
        "n_train": int(len(X_tr)),
        "n_test": int(len(X_te)),
    }

phase1_oof = pd.concat(phase1_oof_frames).reset_index(drop=True)
phase1_sub = pd.concat(phase1_test_frames).sort_values("id").reset_index(drop=True)

log.info("=" * 64)
log.info("PHASE 1 DONE.  Tg OOF R^2 = %.4f  |  Egc OOF R^2 = %.4f",
          phase1_summary["per_target"]["tg"]["blend_oof_r2"],
          phase1_summary["per_target"]["egc"]["blend_oof_r2"])
log.info("=" * 64)

## 6. Phase 2 — Chemprop D-MPNN Multitask

Directed message-passing neural network. Shared molecular graph representation feeds
into a two-task regression head (Tg + Egc jointly). 5-fold × 3-seed bag = 15 models.

Runs on GPU if available, else CPU (much slower).

In [ ]:
import torch
try:
    from lightning import pytorch as pl
except ImportError:
    import pytorch_lightning as pl
from chemprop import data as cdata, featurizers as cfeat, models as cmodels, nn as cnn


def lightning_accelerator():
    if torch.cuda.is_available():
        return "gpu", 1
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return "mps", 1
    return "cpu", 1


def long_to_wide(train_df):
    rows = {}
    for smi, t, y in zip(train_df["smiles"].values, train_df["target_type"].values,
                          train_df["target"].values):
        if smi not in rows:
            rows[smi] = {"smiles": smi, "tg": np.nan, "egc": np.nan}
        rows[smi][t] = y
    return pd.DataFrame(list(rows.values()))[["smiles", "tg", "egc"]]


def _apply_wide_transforms(wide):
    out = wide.copy()
    for t in TARGETS:
        y_t, _ = transform_target(wide[t].values, TARGET_TRANSFORMS[t])
        out[t] = y_t
    return out


def _inverse_wide(transformed):
    out = np.empty_like(transformed)
    for ti, t in enumerate(TARGETS):
        _, inv = transform_target(np.array([0.0]), TARGET_TRANSFORMS[t])
        out[:, ti] = inv(transformed[:, ti])
    return out


def _build_mpnn(seed):
    pl.seed_everything(seed)
    mp = cnn.BondMessagePassing(d_h=CP_MP_HIDDEN, depth=CP_MP_DEPTH, dropout=CP_DROPOUT)
    agg = cnn.MeanAggregation()
    pred = cnn.RegressionFFN(input_dim=CP_MP_HIDDEN, hidden_dim=CP_FFN_HIDDEN,
                              n_layers=CP_FFN_DEPTH, dropout=CP_DROPOUT, n_tasks=len(TARGETS))
    return cmodels.MPNN(mp, agg, pred, batch_norm=True)


def _make_loaders(tr_smis, tr_z, va_smis, va_z):
    feat = cfeat.SimpleMoleculeMolGraphFeaturizer()
    tr_dps = [cdata.MoleculeDatapoint.from_smi(s, y=y.astype(np.float32))
              for s, y in zip(tr_smis, tr_z)]
    va_dps = [cdata.MoleculeDatapoint.from_smi(s, y=y.astype(np.float32))
              for s, y in zip(va_smis, va_z)]
    return (
        cdata.build_dataloader(cdata.MoleculeDataset(tr_dps, feat),
                                batch_size=CP_BATCH_SIZE, num_workers=0),
        cdata.build_dataloader(cdata.MoleculeDataset(va_dps, feat),
                                batch_size=CP_BATCH_SIZE, num_workers=0, shuffle=False),
    )


def _make_predict_loader(smis):
    feat = cfeat.SimpleMoleculeMolGraphFeaturizer()
    dps = [cdata.MoleculeDatapoint.from_smi(s) for s in smis]
    return cdata.build_dataloader(cdata.MoleculeDataset(dps, feat),
                                    batch_size=CP_BATCH_SIZE, num_workers=0, shuffle=False)


def train_chemprop_multitask(train_df, test_df):
    acc, dev = lightning_accelerator()
    log.info("Chemprop accelerator: %s (%d)", acc, dev)

    wide_orig = long_to_wide(train_df)
    wide_t = _apply_wide_transforms(wide_orig)
    test_smis = test_df["smiles"].drop_duplicates().tolist()
    train_smis = wide_orig["smiles"].values
    ys = wide_t[TARGETS].values.astype(np.float32)
    strat = np.nanmean(ys, axis=1)
    folds = stratified_quantile_split(strat, n_folds=CP_N_FOLDS,
                                        n_bins=N_QUANTILE_BINS, seed=SEED)

    log.info("Chemprop training set: %d unique SMILES", len(wide_orig))
    log.info("  has tg  labeled: %d", wide_orig["tg"].notna().sum())
    log.info("  has egc labeled: %d", wide_orig["egc"].notna().sum())

    oof_t = np.full((len(wide_orig), len(TARGETS)), np.nan, dtype=np.float64)
    test_acc_t = np.zeros((len(test_smis), len(TARGETS)), dtype=np.float64)
    n_models = 0

    for fold_idx, (tr_idx, va_idx) in enumerate(folds):
        log.info("-" * 64)
        log.info("Chemprop FOLD %d/%d", fold_idx + 1, CP_N_FOLDS)
        tr_smis_f, va_smis_f = train_smis[tr_idx], train_smis[va_idx]
        tr_ys, va_ys = ys[tr_idx], ys[va_idx]
        mu = np.nanmean(tr_ys, axis=0).astype(np.float32)
        sd = np.nanstd(tr_ys, axis=0).astype(np.float32)
        sd = np.where(sd < 1e-6, 1.0, sd)
        tr_z, va_z = (tr_ys - mu) / sd, (va_ys - mu) / sd

        seed_val, seed_test = [], []
        for seed in CP_BAG_SEEDS:
            try:
                tr_loader, va_loader = _make_loaders(tr_smis_f, tr_z, va_smis_f, va_z)
                model = _build_mpnn(seed)
                trainer = pl.Trainer(
                    accelerator=acc, devices=dev, max_epochs=CP_MAX_EPOCHS,
                    enable_progress_bar=False, enable_checkpointing=False, logger=False,
                    callbacks=[pl.callbacks.EarlyStopping(monitor="val_loss",
                                                            patience=CP_PATIENCE, mode="min",
                                                            check_finite=False)],
                    gradient_clip_val=1.0,
                )
                t0 = time.time()
                trainer.fit(model, tr_loader, va_loader)
                val_z = torch.cat(trainer.predict(model, _make_predict_loader(va_smis_f.tolist())),
                                    dim=0).cpu().numpy()
                test_z = torch.cat(trainer.predict(model, _make_predict_loader(test_smis)),
                                     dim=0).cpu().numpy()
                seed_val.append(val_z * sd + mu)
                seed_test.append(test_z * sd + mu)
                log.info("[fold%d seed=%d] done in %.1fs",
                         fold_idx + 1, seed, time.time() - t0)
            except Exception as e:
                log.exception("[fold%d seed=%d] failed: %s", fold_idx + 1, seed, e)

        if not seed_val:
            continue
        oof_t[va_idx] = np.mean(np.stack(seed_val), axis=0)
        test_acc_t += np.mean(np.stack(seed_test), axis=0) * len(seed_val)
        n_models += len(seed_val)
        log.info("[fold %d] done. bagged %d models", fold_idx + 1, n_models)

    if n_models == 0:
        raise RuntimeError("Chemprop training failed on every fold")
    log.info("Chemprop total models bagged: %d", n_models)
    return oof_t, test_acc_t / n_models, wide_orig["smiles"].values.tolist(), test_smis, n_models

In [ ]:
t_cp_start = time.time()
oof_t_all, test_t_all, wide_smis, test_unique, n_chemprop_models = train_chemprop_multitask(train, test)
log.info("Chemprop training total: %.1f min (%d models)",
         (time.time() - t_cp_start) / 60, n_chemprop_models)

# Convert wide (unique SMILES) predictions back to long-format aligned with train.csv / test.csv rows
oof_orig = _inverse_wide(oof_t_all)
test_orig = _inverse_wide(test_t_all)
has_oof = ~np.isnan(oof_t_all).any(axis=1)

smi_to_idx_train = {s: i for i, s in enumerate(wide_smis)}
smi_to_idx_test = {s: i for i, s in enumerate(test_unique)}

chemprop_oof_rows = []
for _, r in train.iterrows():
    idx = smi_to_idx_train[r["smiles"]]
    ti = TARGETS.index(r["target_type"])
    chemprop_oof_rows.append({
        "smiles": r["smiles"], "target_type": r["target_type"],
        "target": float(r["target"]),
        "oof_chemprop": float(oof_orig[idx, ti]) if has_oof[idx] else np.nan,
        "has_oof": bool(has_oof[idx]),
    })
chemprop_oof = pd.DataFrame(chemprop_oof_rows)

chemprop_sub_rows = []
for _, r in test.iterrows():
    idx = smi_to_idx_test[r["smiles"]]
    ti = TARGETS.index(r["target_type"])
    chemprop_sub_rows.append({"id": int(r["id"]),
                                "target": float(test_orig[idx, ti])})
chemprop_sub = pd.DataFrame(chemprop_sub_rows).sort_values("id").reset_index(drop=True)

# Per-target Chemprop OOF R^2
for t in TARGETS:
    ti = TARGETS.index(t)
    mask = (~np.isnan(oof_orig[:, ti]))
    # cross-reference to only rows in train that have this target labeled
    df_t = train[train["target_type"] == t]
    smi_to_y = dict(zip(df_t["smiles"].values, df_t["target"].values))
    y_true, y_pred = [], []
    for smi in df_t["smiles"].values:
        idx = smi_to_idx_train[smi]
        if has_oof[idx]:
            y_true.append(smi_to_y[smi])
            y_pred.append(oof_orig[idx, ti])
    if y_true:
        r2 = r2_score(y_true, y_pred)
        log.info("[%s] Chemprop OOF R^2 = %.4f (n=%d)", t, r2, len(y_true))

## 7. NNLS Blend — per-target non-negative least squares over Phase 1 + Chemprop OOF

For each target, fit weights `w1, w2 >= 0` minimizing `‖y − w1·phase1_oof − w2·chemprop_oof‖²`,
normalize to sum-to-one, then apply the same weights to test predictions to produce the
final submission.

In [ ]:
final_test_frames = []
final_oof_frames = []
blend_summary = {"per_target": {}}

for t in TARGETS:
    log.info("-" * 32)
    log.info("Blending %s", t.upper())
    tr_mask = (train["target_type"] == t).values
    te_mask = (test["target_type"] == t).values
    smis_t = train.loc[tr_mask, "smiles"].reset_index(drop=True).values
    y_tr = train.loc[tr_mask, "target"].reset_index(drop=True).values
    ids_te = test.loc[te_mask, "id"].reset_index(drop=True).values

    p1_slice = phase1_oof[phase1_oof["target_type"] == t].reset_index(drop=True)
    cp_slice = chemprop_oof[chemprop_oof["target_type"] == t].reset_index(drop=True)
    assert (p1_slice["smiles"].values == smis_t).all(), f"[{t}] phase1 misaligned"
    assert (cp_slice["smiles"].values == smis_t).all(), f"[{t}] chemprop misaligned"

    p1_oof = p1_slice["oof_blend"].values
    cp_oof = cp_slice["oof_chemprop"].values
    mask = cp_slice["has_oof"].values
    p1_test = phase1_sub.set_index("id").loc[ids_te, "target"].values
    cp_test = chemprop_sub.set_index("id").loc[ids_te, "target"].values

    if mask.sum() < 50:
        log.warning("[%s] Chemprop OOF too sparse — falling back to phase1 only", t)
        blend_test = p1_test
        weights = np.array([1.0, 0.0])
        stack_r2 = float(r2_score(y_tr, p1_oof))
    else:
        A_sub = np.column_stack([p1_oof[mask], cp_oof[mask]]).astype(np.float64)
        y_sub = y_tr[mask].astype(np.float64)
        w_raw, _ = nnls(A_sub, y_sub)
        w_sum = w_raw.sum()
        weights = (w_raw / w_sum) if w_sum > 1e-9 else np.array([1.0, 0.0])
        log.info("[%s] NNLS weights: phase1=%.4f  chemprop=%.4f (raw sum=%.4f)",
                  t, weights[0], weights[1], w_sum)
        stack_oof_sub = A_sub @ weights
        stack_r2 = float(r2_score(y_sub, stack_oof_sub))
        log.info("[%s] per-base subset R^2:  phase1=%.4f  chemprop=%.4f",
                  t, r2_score(y_sub, p1_oof[mask]), r2_score(y_sub, cp_oof[mask]))
        log.info("[%s] STACK OOF R^2 (subset, n=%d) = %.4f", t, int(mask.sum()), stack_r2)
        blend_test = weights[0] * p1_test + weights[1] * cp_test

    final_test_frames.append(pd.DataFrame({"id": ids_te, "target": blend_test}))
    final_oof_frames.append(pd.DataFrame({
        "smiles": smis_t, "target_type": t, "target": y_tr,
        "oof_phase1": p1_oof, "oof_chemprop": cp_oof,
        "has_chemprop_oof": mask,
    }))
    blend_summary["per_target"][t] = {
        "phase1_weight": float(weights[0]),
        "chemprop_weight": float(weights[1]),
        "stack_oof_r2_subset": stack_r2,
        "phase1_full_oof_r2": phase1_summary["per_target"][t]["blend_oof_r2"],
    }

# --- Write final submission ---
final_sub = pd.concat(final_test_frames).sort_values("id").reset_index(drop=True)
sub_path = OUT_DIR / "submission.csv"
final_sub.to_csv(sub_path, index=False)

final_oof = pd.concat(final_oof_frames).reset_index(drop=True)
(OUT_DIR / "oof_final.csv").parent.mkdir(parents=True, exist_ok=True)
final_oof.to_csv(OUT_DIR / "oof_final.csv", index=False)

summary = {
    "phase1_per_target": phase1_summary["per_target"],
    "blend_per_target": blend_summary["per_target"],
    "chemprop_n_models": n_chemprop_models,
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))

log.info("=" * 64)
log.info("SUBMISSION WRITTEN: %s (%d rows)", sub_path, len(final_sub))
log.info("=" * 64)

## 8. Verify submission

In [ ]:
print(f"Submission file: {sub_path}")
print(f"Rows: {len(final_sub)}  (should match test.csv: {len(test)})")
print()
print("First 5 rows:")
print(final_sub.head())
print()
print("Distribution by target_type:")
sub_with_type = final_sub.merge(test[["id", "target_type"]], on="id")
for t in TARGETS:
    vals = sub_with_type[sub_with_type["target_type"] == t]["target"]
    print(f"  {t}: n={len(vals)}  mean={vals.mean():.3f}  min={vals.min():.3f}  max={vals.max():.3f}")

print()
print("Per-target OOF R^2 (Phase 1 solo):")
for t in TARGETS:
    print(f"  {t}: {phase1_summary['per_target'][t]['blend_oof_r2']:.4f}")
print()
print("NNLS blend weights:")
for t in TARGETS:
    w = blend_summary["per_target"][t]
    print(f"  {t}:  phase1={w['phase1_weight']:.4f}  chemprop={w['chemprop_weight']:.4f}")
print()
print(f"Total runtime: {(time.time() - t_start) / 60:.1f} min")
print(f"Submission ready at: {sub_path}")